# Lecture 6: Eigenvalues and eigenvectors

In [ ]:
# Setup. Requirements: pip install numpy scipy matplotlib networkx
import numpy as np
import scipy.linalg as sla
import scipy.sparse as sp
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import YouTubeVideo

%matplotlib inline
plt.rcParams.update({'figure.figsize': (7, 4), 'figure.dpi': 110})
rng = np.random.default_rng(0)


## Recap of the previous lecture
- Linear systems
- Gaussian elimination
- LU decomposition
- Condition number as a measure of the sensitivity of the problem to perturbations of the data (conditioning is a property of the problem, stability is a property of the algorithm)

## Today's lecture
Today we will talk about:
- Eigenvectors and their applications (PageRank)
- Gershgorin circles
- Computing eigenvectors using the power method
- Schur theorem
- Normal matrices

## What is an eigenvector?

- **Definition.** A vector $x \ne 0$ is called an **eigenvector** of a square matrix $A$ if there exists a number $\lambda$ such that

$$ Ax = \lambda x. $$

- The number $\lambda$ is called an **eigenvalue**. The pair $(\lambda, x)$ is called an **eigenpair**.

- Since $A - \lambda I$ must have a non-trivial kernel, eigenvalues are the roots of the **characteristic polynomial**

$$ p(\lambda) = \det (A - \lambda I) = 0. $$

- $p$ has degree $n$, so over $\mathbb{C}$ an $n\times n$ matrix has exactly $n$ eigenvalues counted with multiplicity. Even for a real $A$ they can be complex: the rotation by $90^\circ$, $\begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$, has $\lambda = \pm i$.

### What does an eigenvector look like?

Take a unit vector $x$ and look at $Ax$. In general $A$ both stretches and **turns** $x$. An eigenvector is a direction that $A$ does not turn: $x$ and $Ax$ lie on the same line through the origin, and $\lambda$ is the stretching factor (a negative $\lambda$ flips the direction).

Below: the unit circle (blue), its image under $A$ (red ellipse) and arrows $x \to Ax$ for several $x$. Only along the dashed eigen-directions does the arrow point along the line through the origin. Note that the two eigen-directions are **not orthogonal**: $A$ is not symmetric.

In [ ]:
A = np.array([[1.0, 1.0],
              [0.5, 1.5]])               # eigenvalues 2 and 0.5
phi = np.linspace(0, 2 * np.pi, 400)
circle = np.vstack([np.cos(phi), np.sin(phi)])
lam, S = np.linalg.eig(A)
lam = np.re(lam)
S = np.re(S)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(*circle, color='tab:blue', lw=1, label=r'$x$, $\|x\|_2=1$')
ax.plot(*(A @ circle), color='tab:red', lw=1, label='$Ax$')
for t in np.linspace(0, 2 * np.pi, 24, endpoint=False):
    x = np.array([np.cos(t), np.sin(t)])
    ax.annotate('', xy=A @ x, xytext=x, arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))
for l, s in zip(lam, S.T):
    ax.plot([-2.5 * s[0], 2.5 * s[0]], [-2.5 * s[1], 2.5 * s[1]], 'k--', lw=1)
    ax.text(*(1.15 * l * s + 0.1), rf'$\lambda={l:.1f}$')
ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=8)
plt.show()


## Eigendecomposition
If a matrix $A$ of size $n\times n$ has $n$ **linearly independent** eigenvectors $s_i$, $i=1,\dots,n$:

$$ As_i = \lambda_i s_i, $$

then this can be written as

$$ A S = S \Lambda, \quad\text{where}\quad S=(s_1,\dots,s_n), \quad \Lambda = \operatorname{diag}(\lambda_1, \dots, \lambda_n), $$

or, since $S$ is invertible,

$$ A = S\Lambda S^{-1}. $$

- This is called the **eigendecomposition** of a matrix. Matrices that admit an eigendecomposition are called **diagonalizable**.

### Existence

- Which matrices are diagonalizable?

- A simple sufficient condition: all eigenvalues are distinct.
- In general (over $\mathbb{C}$), a matrix is diagonalizable iff for each eigenvalue its **algebraic multiplicity** (multiplicity as a root of the characteristic polynomial) equals its **geometric multiplicity** (dimension of the eigenspace $\ker(A-\lambda I)$).

- For our purposes the most important class is the class of **normal matrices**:

$$AA^* = A^* A,$$

where $A^* = \overline{A}^\top$ is the conjugate transpose. Normal matrices are not just diagonalizable: they are diagonalizable by a **unitary** matrix, $A = U\Lambda U^*$.

- We will prove this after the Schur decomposition.

#### Example

The matrix

$$A = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}$$

has one eigenvalue $1$ of algebraic multiplicity $2$ (its characteristic polynomial is $p(\lambda)=(1-\lambda)^2$), but all its eigenvectors have the form $\begin{pmatrix} c \\ 0 \end{pmatrix}$, $c\ne0$: the geometric multiplicity is $1$, and the matrix is not diagonalizable.

Let us see what `numpy` returns.

In [ ]:
A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
lam, S = np.linalg.eig(A)
print('eigenvalues:', lam)
print('eigenvectors (columns):')
print(S)
print(f'cond(S) = {np.linalg.cond(S):.1e}')


The two returned columns are parallel up to rounding: $S$ is singular to machine precision (huge $\operatorname{cond}(S)$), so $A = S\Lambda S^{-1}$ is meaningless here.

## Pause: are eigenvalues additive?

$A = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}$ and $B = \begin{pmatrix} 0 & 0 \\ 1 & 0 \end{pmatrix}$ both have all eigenvalues equal to $0$ (they are triangular with zero diagonal).

**Question.** What are the eigenvalues of $A + B$? Of $AB$?

In [ ]:
A = np.array([[0.0, 1.0], [0.0, 0.0]])
B = np.array([[0.0, 0.0], [1.0, 0.0]])
for name, M in [('A + B', A + B), ('A B', A @ B), ('B A', B @ A)]:
    print(f'{name:6s} eigenvalues: {np.linalg.eigvals(M)}')


- $\lambda(A+B) = \pm 1$, although all eigenvalues of $A$ and $B$ are $0$; and $\lambda(AB) = \{1, 0\}$.
- The eigenvalues of a sum or a product are **not** determined by the eigenvalues of the terms: they depend on how the eigenvectors of $A$ and $B$ are positioned relative to each other. Only if $A$ and $B$ share a basis of eigenvectors (e.g. commuting diagonalizable matrices) do eigenvalues add and multiply.
- What is always true: $\operatorname{tr}(A+B) = \operatorname{tr}A + \operatorname{tr}B$, i.e. the **sum** of all eigenvalues is additive; and $AB$ and $BA$ have the same eigenvalues.

## Why are eigenvectors and eigenvalues important?

- Eigenvectors are both an important auxiliary tool and play an important role in applications.


## Applications of eigenvalues/eigenvectors

- **Communication theory:** theoretical limit on the amount of information transmitted through a channel
- **Designing bridges** and other structures (mechanical engineering)
- **Designing** hi-fi audio systems
- **Quantum chemistry:** the microworld is governed by the **Schrödinger equation**, which is an eigenvalue problem:

$$ H \psi = E \psi, $$

where $H$ is the Hamiltonian (the energy operator), $\psi$ is the wave function and $E$ is the energy level.
- Model order reduction of complex systems
- Graph analysis (PageRank, graph clustering)

## Eigenvalues are vibrational frequencies

A typical computation of eigenvalues / eigenvectors is for studying

- vibrations of mechanical structures,
- model order reduction of complex systems.

For a structure with mass matrix $M$ and stiffness matrix $K$ (both symmetric positive definite) small vibrations obey $M\ddot x + Kx = 0$, where $x(t)$ is the vector of displacements. Looking for solutions $x(t) = v\cos(\omega t)$ gives

$$ K v = \omega^2 M v: $$

the eigenvalues are the squared natural frequencies $\omega^2$, the eigenvectors $v$ are the vibration modes. An external force oscillating near one of these frequencies causes resonance.

In [ ]:
# YouTubeVideo('VcCcMZo6J6w')


## Google PageRank
- One of the most famous eigenvector computations is **Google PageRank**.
- It is not the main ranking signal of Google nowadays, but it was one of the key features in its early days. The question is how to rank web pages: which ones are important and which are not.
- All we know about the web is which page links to which. PageRank is defined recursively.
- Denote by $p_i$ the **importance** of the $i$-th page. A page is important if important pages link to it: each page $j$ splits its importance equally among its $L(j)$ outgoing links,

$$ p_i = \sum_{j \in N(i)} \frac{p_j}{L(j)}, $$

where $L(j)$ is the number of outgoing links of page $j$ and $N(i)$ is the set of pages that link **to** page $i$. In matrix form

$$ p = G p, \qquad G_{ij} = \begin{cases} 1/L(j), & j \text{ links to } i,\\ 0, & \text{otherwise,}\end{cases} $$

or as an eigenvalue problem

$$
   Gp = 1 \cdot p,
$$

i.e. the eigenvalue $1$ is already known, and we look for the eigenvector. If every page has at least one outgoing link, $G$ is **left stochastic**: its entries are nonnegative and its columns sum up to $1$.

**Exercise.** Check that for any left stochastic matrix $1$ is an eigenvalue and all eigenvalues satisfy $|\lambda|\le 1$, i.e. the spectral radius $\rho(G) = \max_i|\lambda_i|$ equals $1$.

<details><summary><b>Hint</b></summary>

$\mathbf 1^\top G = \mathbf 1^\top$, where $\mathbf 1$ is the vector of all ones, so $1$ is an eigenvalue of $G^\top$, hence of $G$. And $|\lambda| \le \|G\|_1 = \max_j \sum_i |G_{ij}| = 1$.

</details>

## Two problems and the damping factor

- **Dangling pages** (no outgoing links): $L(j)=0$, the column of $G$ is zero, and $G$ is not stochastic.
- **Non-uniqueness:** if the graph is disconnected, the eigenvalue $1$ is multiple, and the ranking is not defined uniquely.

The fix used by PageRank: a random surfer follows a link with probability $\alpha$ and jumps to a random page with probability $1-\alpha$ (typically $\alpha = 0.85$). Zero columns are replaced by $\frac1n\mathbf 1$, and

$$ G_\alpha = \alpha G + \frac{1-\alpha}{n}\,\mathbf 1\mathbf 1^\top. $$

$G_\alpha$ is positive, so by the **Perron–Frobenius theorem** the eigenvalue $1$ is simple, and its eigenvector can be chosen positive. Moreover, all other eigenvalues satisfy $|\lambda| \le \alpha$; this will be important for the power method.

## Demo
- We can compute PageRank using Python packages.
- We will use the `networkx` package for working with graphs:

```
pip install networkx
```

- Other packages to work with graphs in Python are [igraph](https://igraph.org/) and [graph-tool](https://graph-tool.skewed.de/).
- They can be useful in your projects.

- We will use a simple example: [Zachary's karate club network](https://en.wikipedia.org/wiki/Zachary%27s_karate_club).
- The data were collected manually in 1977; this is a classical social network dataset (34 members, 78 friendship ties). It is built into `networkx`, no file is needed.

In [ ]:
graph = nx.karate_club_graph()
pos = nx.spring_layout(graph, seed=1)          # fixed layout: the same picture on every run
fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx(graph, pos, ax=ax, node_color='tab:red', font_size=8)
ax.set_axis_off()
plt.show()
print(graph)


- Now we compute PageRank with the built-in `networkx` function.
- The node size is proportional to its PageRank.
- `weight=None`: in recent `networkx` versions the edges of this graph carry weights (number of joint activities); we use the plain link structure, as in the formula above.

In [ ]:
pr = nx.pagerank(graph, alpha=0.85, weight=None, tol=1e-12, max_iter=1000)
p_nx = np.array([pr[v] for v in graph])

fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx(graph, pos, ax=ax, node_size=8000 * p_nx, node_color='tab:red', font_size=8)
ax.set_axis_off()
plt.show()

top = np.argsort(p_nx)[::-1][:5]
print('top-5 nodes:', top, np.round(p_nx[top], 3))


## Our own PageRank

PageRank is computed by the iteration $p_{k+1} = G_\alpha p_k$ (this is the **power method**, see below). We never form the dense matrix $G_\alpha$: if $\mathbf 1^\top p_k = 1$, then

$$ G_\alpha p_k = \alpha G p_k + \frac{1-\alpha}{n}\mathbf 1, $$

and $G$ is sparse.

In [ ]:
def link_matrix(graph: nx.Graph) -> sp.csr_array:
    # column-stochastic G: G[i, j] = 1 / L(j) if j links to i
    A = nx.to_scipy_sparse_array(graph, weight=None, format='csr')   # A[j, i] = 1 if j links to i
    out_deg = np.asarray(A.sum(axis=1)).ravel()                     # L(j)
    assert np.all(out_deg > 0), 'dangling pages are not handled here'
    return sp.csr_array(A.T.multiply(1.0 / out_deg[None, :]))       # column j divided by L(j)


def pagerank(graph: nx.Graph, alpha: float = 0.85, tol: float = 1e-12, max_iter: int = 10_000):
    G = link_matrix(graph)
    n = G.shape[0]
    p = np.full(n, 1.0 / n)
    for k in range(1, max_iter + 1):
        p_new = alpha * (G @ p) + (1 - alpha) / n
        if np.abs(p_new - p).sum() < tol:
            return p_new, k
        p = p_new
    raise RuntimeError('the power method did not converge')


p_ours, iters = pagerank(graph)
print(f'{iters} iterations, max |ours - networkx| = {np.abs(p_ours - p_nx).max():.1e}')


**Remark.** On an **undirected** connected graph with $\alpha = 1$ the solution is known explicitly: $p_i = \deg(i)/\sum_j \deg(j)$, where $\deg(i)$ is the number of edges at node $i$ (check that it satisfies $p = Gp$). So on the karate club PageRank is close to the normalised degree; it becomes really informative on directed graphs such as the web.

In [ ]:
deg = np.array([d for _, d in graph.degree()], dtype=float)
p1, _ = pagerank(graph, alpha=1.0)
print(f'alpha = 1:    max |p - deg / sum(deg)| = {np.abs(p1 - deg / deg.sum()).max():.1e}')
print(f'alpha = 0.85: correlation of p with degree = {np.corrcoef(p_ours, deg)[0, 1]:.3f}')


## Computations of eigenvalues

- How to compute eigenvalues and eigenvectors?

There are two types of eigenproblems:

- **full eigenproblem:** all eigenvalues and eigenvectors are required; $O(n^3)$ operations for a dense matrix;
- **partial eigenproblem:** only some eigenvalues are required: the smallest or the largest ones, or those in a given region of the complex plane (PageRank, vibrations of large structures; huge sparse matrices).

## Computation of the eigenvalues via the characteristic equation
The eigenvalue problem has the form

$$ Ax = \lambda x, $$

or

$$ (A - \lambda I) x = 0, $$

therefore the matrix $A - \lambda I$ has a non-trivial kernel and must be singular.

That means that the **determinant**

$$ p(\lambda) = \det(A - \lambda I) = 0. $$

- This equation is called the **characteristic equation**; $p$ is a polynomial of degree $n$.
- A polynomial of degree $n$ has $n$ complex roots, counted with multiplicity.

## Recall the determinant
The determinant of a square matrix $A$ is defined as

$$\det A = \sum_{\sigma \in S_n} \operatorname{sgn}(\sigma)\prod_{i=1}^n a_{i, \sigma(i)},$$

where
- $S_n$ is the set of all **permutations** $\sigma$ of the numbers $1, \ldots, n$ (there are $n!$ of them);
- $\operatorname{sgn}(\sigma) = (-1)^p$ is the **sign** of the permutation, where $p$ is the number of transpositions (swaps of two elements) that compose $\sigma$; the parity of $p$ does not depend on the choice of transpositions.

## Properties of the determinant
The determinant has many nice properties:

1. $\det(AB) = \det(A) \det(B)$.
2. It is linear in each row: if one row is a sum of two vectors, the determinant is the sum of two determinants.
3. **Minor (Laplace) expansion:** the determinant can be expanded along any row or column.

- Computed via the definition or via recursive minor expansion, the determinant costs about $n!$ operations, which grows faster than any exponential.

- Can we do it in $\mathcal{O}(n^3)$?

Yes: by the LU decomposition with pivoting from the previous lecture. If $PA = LU$ with a permutation matrix $P$ and unit lower triangular $L$, then

$$ \det A = \det(P^\top)\det L \det U = \pm\prod_{i=1}^n u_{ii}, $$

where the sign is the sign of the permutation. This is how `np.linalg.det` works.

## Eigenvalues and the characteristic equation

- Now we go back to the eigenvalues.

- The characteristic equation suggests a **naïve** algorithm:

$$p(\lambda) = \det(A - \lambda I)$$

1. Compute the coefficients of the polynomial.
2. Compute its roots.

**Is this a good idea?**

**Give your feedback.**

Let us do a short demo with the matrix $a_{ji} = \dfrac{1}{i - j + 0.5}$, $n = 40$. We compare the eigenvalues computed by LAPACK (`np.linalg.eigvals`) with the roots of the characteristic polynomial (`np.poly` gives the coefficients, `np.roots` the roots).

In [ ]:
n = 40
idx = np.arange(n)
a = 1.0 / (idx[None, :] - idx[:, None] + 0.5)   # a[j, i] = 1 / (i - j + 0.5)

ev_lapack = np.linalg.eigvals(a)
coeffs = np.poly(a)                              # coefficients of det(lambda I - a)
ev_roots = np.roots(coeffs)

fig, ax = plt.subplots()
ax.scatter(ev_roots.real, ev_roots.imag, marker='x', label='roots of $p$')
ax.scatter(ev_lapack.real, ev_lapack.imag, marker='o', facecolors='none', edgecolors='tab:orange', label='LAPACK')
ax.set_xlabel('real part'); ax.set_ylabel('imaginary part')
ax.legend(loc='best')
plt.show()


A detail that makes the demo even more convincing: for a matrix argument `np.poly` computes the coefficients **from the eigenvalues** (it calls `eigvals` and expands $\prod_i(\lambda-\lambda_i)$). So the coefficients come from accurate eigenvalues and are only rounded to double precision, and still the roots are far from the eigenvalues.

### Moral

- Do not do that, unless you have a reason.
- Polynomial rootfinding in the monomial basis $\{1, x, x^2, \dots\}$ is very **ill-conditioned** (in other bases, e.g. Chebyshev polynomials, it can be much better). The Gram matrix of the monomials on $[0, 1]$,

$$h_{ij} = \int_0^1 x^i x^j\, dx = \frac{1}{i+j+1}, \qquad i, j = 0, \dots, n-1,$$

is the Hilbert matrix, whose singular values decay exponentially.
- So the monomials are almost linearly dependent: very different coefficient vectors give almost the same function on $[0, 1]$, and a tiny change of the coefficients can move the roots a lot.

### One more classical example: the Wilkinson polynomial

$p(x) = (x-1)(x-2)\cdots(x-20)$: the roots are well-separated integers. We store the coefficients in double precision and compute the roots.

In [ ]:
roots_true = np.arange(1, 21, dtype=float)
coeffs = np.poly(roots_true)          # coefficients of (x-1)...(x-20), rounded to double
# coeffs[1] -= 1e-10
roots = np.roots(coeffs)

fig, ax = plt.subplots()
ax.plot(roots_true, np.zeros_like(roots_true), 'o', mfc='none', label='exact roots')
ax.plot(roots.real, roots.imag, 'x', color='tab:red', label='computed roots')
ax.set_xlabel('Re'); ax.set_ylabel('Im'); ax.legend()
plt.show()
print(f'largest coefficient: {np.abs(coeffs).max():.1e}')
print(f'max root error:      {np.abs(np.sort_complex(roots) - roots_true).max():.1e}')


The problem is not the root finder (`np.roots` itself computes eigenvalues of the companion matrix) but the representation: the map *coefficients → roots* is ill-conditioned. For the matrix $\operatorname{diag}(1,\dots,20)$ itself the eigenvalue problem is perfectly conditioned. Eigenvalue algorithms work with the matrix directly and never form $p(\lambda)$.

## Gershgorin circles
- There is a very interesting theorem that often helps to localise the eigenvalues.

- It is called the **Gershgorin theorem** (1931).

- It states that all eigenvalues $\lambda_i$, $i = 1, \ldots, n$, are located in the union of the **Gershgorin circles** (discs) $C_i$, where $C_i$ is the disc in the complex plane with centre $a_{ii}$ and radius

$$r_i = \sum_{j \ne i} |a_{ij}|,$$

the sum of moduli of the off-diagonal entries of row $i$.

- Moreover, a disc that does not intersect the other discs contains exactly one eigenvalue (more generally, $k$ discs forming a connected region disjoint from the other discs contain exactly $k$ eigenvalues).
- The proof is instructive, since it uses the concepts from the previous lectures.

## Proof
First, we show that if the matrix $A$ is **strictly diagonally dominant**, i.e.

$$
   |a_{ii}| > \sum_{j \ne i} |a_{ij}| = r_i, \qquad i = 1, \dots, n,
$$

then it is nonsingular.

We separate the diagonal part $D$ and the off-diagonal part $S$:

$$
    A = D + S = D( I + D^{-1}S).
$$

$D$ is invertible, since $|a_{ii}| > r_i \ge 0$. Row $i$ of $D^{-1}S$ is row $i$ of $S$ divided by $a_{ii}$, so

$$ \|D^{-1}S\|_\infty = \max_i \frac{r_i}{|a_{ii}|} < 1, $$

where $\|B\|_\infty = \max_i \sum_j |b_{ij}|$ is the maximal absolute **row** sum: the radii are row sums, so this is the right norm. Then the **Neumann series** $(I + D^{-1}S)^{-1} = \sum_{k\ge0} (-D^{-1}S)^k$ converges, the matrix $I + D^{-1}S$ is invertible, and hence $A$ is invertible.

Now the proof goes by contradiction:

- if an eigenvalue $\lambda$ lies outside all the discs, then $|a_{ii} - \lambda| > r_i$ for every $i$, i.e. the matrix $A - \lambda I$ is strictly diagonally dominant (its off-diagonal part is the same as that of $A$);
- thus it is invertible;
- that means that $(A - \lambda I) x = 0$ implies $x = 0$, and $\lambda$ is not an eigenvalue. $\blacksquare$

**Another proof in one line.** Let $Ax = \lambda x$ and $|x_i| = \max_j|x_j|$. The $i$-th equation gives $|\lambda - a_{ii}|\,|x_i| = \big|\sum_{j\ne i}a_{ij}x_j\big| \le r_i|x_i|$, so $\lambda\in C_i$: the eigenvalue lies in the disc of the row where its eigenvector is largest.

### Remarks

- Applying the theorem to $A^\top$ (same eigenvalues) gives discs with **column** sums as radii; both sets of discs are valid.
- If $k$ discs form a connected region disjoint from the other discs, that region contains exactly $k$ eigenvalues. Idea: let $D$ be the diagonal part of $A$ and $A(t) = D + t(A-D)$, $t\in[0,1]$; the eigenvalues move continuously from the centres and cannot jump between disjoint regions.
- For a matrix close to diagonal the discs are small, and the diagonal entries are good approximations of the eigenvalues.

In [ ]:
def plot_gershgorin(A: np.ndarray, ax) -> None:
    centers = np.diag(A)
    radii = np.abs(A).sum(axis=1) - np.abs(centers)
    for c, r in zip(centers, radii):
        ax.add_patch(plt.Circle((c.real, c.imag), r, alpha=0.2, ec='tab:blue'))
    ev = np.linalg.eigvals(A)
    ax.plot(ev.real, ev.imag, 'x', color='tab:red', ms=8, label='eigenvalues')
    ax.plot(centers.real, centers.imag, '.', color='k', label='centres $a_{ii}$')
    ax.set_xlim((centers.real - radii).min() - 1, (centers.real + radii).max() + 1)
    ax.set_ylim(-radii.max() - 1, radii.max() + 1)
    ax.set_aspect('equal')
    ax.legend(fontsize=8)


A = np.array([[5.0, 1.0, 1.0],
              [1.0, 0.0, 0.5],
              [2.0, 0.0, 10.0]])
A_noisy = A + 2 * rng.standard_normal(A.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_gershgorin(A, axes[0]); axes[0].set_title('A')
plot_gershgorin(A_noisy, axes[1]); axes[1].set_title('A + noise')
plt.show()


Left: the three discs (centres $5$, $0$, $10$, radii $2$, $1.5$, $2$) are disjoint, so each contains exactly one eigenvalue. Right: after adding noise the discs grow and may overlap; the theorem still holds for their union.

### Two traps with Gershgorin discs

**1. A disc may contain no eigenvalue.** The theorem says that every eigenvalue lies in *some* disc, not that every disc contains an eigenvalue. For

$$ A = \begin{pmatrix} 0 & 1 \\ 4 & 0 \end{pmatrix} $$

the discs are $|z| \le 1$ and $|z| \le 4$, while $\lambda = \pm 2$: the small disc is empty. The counting statement does not help here: the two discs are not disjoint.

**2. The discs depend on the basis, the eigenvalues do not.** For a diagonal $D$ with positive entries, $D^{-1}AD$ has the same eigenvalues but different discs. With $D = \operatorname{diag}(1, 2)$ we get $D^{-1}AD = \begin{pmatrix} 0 & 2 \\ 2 & 0 \end{pmatrix}$: both discs are $|z|\le2$, and the eigenvalues lie exactly on their boundary. Choosing $D$ well is a cheap way to sharpen the bound.

In [ ]:
A = np.array([[0.0, 1.0], [4.0, 0.0]])
D = np.diag([1.0, 2.0])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_gershgorin(A, axes[0]); axes[0].set_title('A')
plot_gershgorin(np.linalg.solve(D, A @ D), axes[1]); axes[1].set_title(r'$D^{-1} A D$')
plt.show()


**Note.** There are more complicated regions that also contain the spectrum, e.g. the **[Cassini ovals](https://en.wikipedia.org/wiki/Cassini_oval)**

$$ C_{ij} = \{z\in\mathbb{C}: |a_{ii} - z|\cdot |a_{jj} - z|\leq r_i r_j\}, \qquad r_i = \sum_{l\ne i} |a_{il}|. $$

Every eigenvalue lies in the union of $C_{ij}$ over all pairs $i \ne j$ (Brauer); this union is contained in the union of the Gershgorin discs, so the localisation is sharper.

## Power method

- We are often interested in computing a part of the spectrum, e.g. the largest or the smallest eigenvalues.
- Also note that the eigenvalues of a Hermitian matrix ($A = A^*$) are always real (prove it!).
- The power method is the simplest method for computing **the eigenvalue of largest modulus**.
- It is also our first example of an **iterative method** and of a **Krylov method**.

<details><summary><b>Why the eigenvalues of a Hermitian matrix are real</b></summary>

Let $Ax = \lambda x$, $x\ne0$. Then $\lambda\, x^*x = x^*Ax$, and $x^*Ax$ is real: $\overline{x^*Ax} = (x^*Ax)^* = x^*A^*x = x^*Ax$. Since $x^*x > 0$, $\lambda$ is real.

</details>

## Power method as a fixed-point iteration

- The eigenvalue problem

$$Ax = \lambda x, \qquad \Vert x \Vert_2 = 1 \ \text{(normalisation for stability)},$$

can be rewritten (for $\lambda > 0$) as the **fixed-point equation** $x = \dfrac{Ax}{\|Ax\|_2}$.
- The corresponding iteration is the **power method**; it finds the eigenvalue of largest modulus of $A$.

## Power method: the algorithm

The iteration is

$$ x_{k+1} = \frac{A x_k}{\|A x_k\|_2}, \qquad k = 0, 1, \dots, $$

and the eigenvalue is approximated by the **Rayleigh quotient**

$$ \lambda^{(k)} = \frac{(A x_k, x_k)}{(x_k, x_k)} = x_k^* A x_k \qquad (\|x_k\|_2 = 1). $$

Each step costs one matrix–vector product: the method works for huge sparse matrices that we cannot even store densely.

- $\lambda^{(k)}$ is not needed for the next iteration, but it gives an error estimate via the **residual** $\|Ax_k - \lambda^{(k)}x_k\|_2$.
- The convergence is geometric: the error decreases as $q^k$, $q = |\lambda_2/\lambda_1| < 1$, where the eigenvalues are ordered by modulus, $|\lambda_1| > |\lambda_2| \ge \dots \ge |\lambda_n|$.
- Hence the convergence can be **arbitrarily slow**: already for $A = \operatorname{diag}(1, 1 - \delta)$ we have $q = 1 - \delta$, and about $1/\delta$ iterations are needed to reduce the error by a factor of $e$.

### Convergence analysis

For a Hermitian $A$ there is an orthonormal basis of eigenvectors (we prove it below, normal matrices). The argument needs only a basis of eigenvectors, so let $A$ be diagonalizable, $A s_i = \lambda_i s_i$, with $|\lambda_1| > |\lambda_2| \ge \dots \ge |\lambda_n|$, and expand $x_0 = \sum_i c_i s_i$ with $c_1 \ne 0$. Then

$$ A^k x_0 = \sum_{i=1}^n c_i \lambda_i^k s_i = c_1\lambda_1^k\Big( s_1 + \sum_{i\ge2}\frac{c_i}{c_1}\Big(\frac{\lambda_i}{\lambda_1}\Big)^k s_i\Big). $$

$x_k$ is $A^kx_0$ normalised, so its direction converges to $s_1$ with error $O\big(|\lambda_2/\lambda_1|^k\big)$: **linear convergence** with ratio $q = |\lambda_2/\lambda_1|$.

**Careful: the direction converges, the vector need not.** After normalisation

$$ x_k = \frac{c_1}{|c_1|}\Big(\frac{\lambda_1}{|\lambda_1|}\Big)^k \frac{s_1 + O(q^k)}{\|s_1 + O(q^k)\|_2}. $$

The factor in front has modulus $1$ but need not converge: for $\lambda_1 < 0$ it alternates between $\pm1$, for complex $\lambda_1$ it rotates. What converges is the line spanned by $x_k$, and the Rayleigh quotient. The assumption $c_1 \ne 0$ is essential, see the pause below.

For a Hermitian $A$ ($A=A^*$) the Rayleigh quotient converges twice as fast, $|\lambda^{(k)} - \lambda_1| = O(q^{2k})$: its error is quadratic in the error of the vector.

In [ ]:
def power_method(A, x0: np.ndarray, num_iter: int):
    # returns the histories of Rayleigh quotients and of normalised vectors
    x = x0 / np.linalg.norm(x0)
    lams, xs = [], []
    for _ in range(num_iter):
        y = A @ x
        x = y / np.linalg.norm(y)
        lams.append(np.vdot(x, A @ x))       # x^* A x
        xs.append(x)
    return np.array(lams), np.array(xs)


n = 100
Q, _ = np.linalg.qr(rng.standard_normal((n, n)))
lam_true = np.concatenate([[1.0, 0.9], rng.uniform(-0.8, 0.8, n - 2)])
A = Q @ np.diag(lam_true) @ Q.T              # symmetric with known eigenvalues
s1 = Q[:, 0]

num_iter = 100
lams, xs = power_method(A, rng.standard_normal(n), num_iter)
k = np.arange(1, num_iter + 1)
err_vec = np.sqrt(np.maximum(1 - (xs @ s1) ** 2, 0))   # sine of the angle between x_k and s_1
err_lam = np.abs(lams.real - lam_true[0])

fig, ax = plt.subplots()
ax.semilogy(k, err_vec, label=r'$\sin\angle(x_k, s_1)$')
ax.semilogy(k, err_lam, label=r'$|\lambda^{(k)} - \lambda_1|$')
ax.semilogy(k, 0.9 ** k, 'k--', lw=1, label=r'$0.9^k$')
ax.semilogy(k, 0.81 ** k, 'k:', lw=1, label=r'$0.81^k$')
ax.set_xlabel('iteration $k$')
ax.legend()
plt.show()


Both errors are straight lines on the semilog plot (linear convergence): the vector with ratio $0.9$, the Rayleigh quotient of this symmetric matrix with ratio $0.9^2 = 0.81$.

## Pause: a bad starting vector

The analysis assumed $c_1 \ne 0$. Take a symmetric matrix with $\lambda_1 = 1$, $\lambda_2 = 0.5$, $|\lambda_i| \le 0.4$ for $i \ge 3$, and start from $x_0 = s_2 + s_3$, which is orthogonal to $s_1$ (up to rounding).

**Question.** In exact arithmetic the iterates converge to $s_2$. What happens on a computer?

In [ ]:
n = 100
Q, _ = np.linalg.qr(rng.standard_normal((n, n)))
lam_true = np.concatenate([[1.0, 0.5], rng.uniform(-0.4, 0.4, n - 2)])
A = Q @ np.diag(lam_true) @ Q.T
x0 = Q[:, 1] + Q[:, 2]                       # orthogonal to s_1 = Q[:, 0]

num_iter = 120
lams, xs = power_method(A, x0, num_iter)
k = np.arange(1, num_iter + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(k, np.abs(xs @ Q[:, 0]), label=r'$|(x_k, s_1)|$')
axes[0].semilogy(k, np.abs(xs @ Q[:, 1]), label=r'$|(x_k, s_2)|$')
axes[0].set_xlabel('$k$'); axes[0].legend()
axes[1].plot(k, lams.real)
axes[1].set_xlabel('$k$'); axes[1].set_title(r'Rayleigh quotient $\lambda^{(k)}$')
plt.show()


- For a few dozen iterations the method happily converges to $\lambda_2 = 0.5$: the Rayleigh quotient is flat, and a naive stopping criterion would stop there.
- Rounding errors create a tiny component along $s_1$, about $10^{-16}$. It doubles at every step ($\lambda_1/\lambda_2 = 2$) and after about $\log_2 10^{16} \approx 53$ steps takes over.
- Two lessons: (1) here rounding errors **help**; with a random $x_0$ we have $c_1 \ne 0$ with probability one anyway; (2) a plateau is not convergence: a stagnating iteration may be just slow.

### Things to remember about the power method

- It finds only the eigenvalue of **largest modulus** and its eigenvector.
- Convergence is linear with ratio $|\lambda_2/\lambda_1|$: slow if $|\lambda_2| \approx |\lambda_1|$.
- If $|\lambda_1| = |\lambda_2|$ but $\lambda_1 \ne \lambda_2$, it may not converge at all. Example: $A = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$ with $\lambda = \pm1$ and $x_0 = (1, 0)^\top$: the iterates alternate between $(1,0)^\top$ and $(0,1)^\top$.
- If $A$ is real and $\lambda_1$ is complex, its conjugate has the same modulus: real iterates do not converge either.
- It needs only products $x \mapsto Ax$: ideal for large sparse matrices.
- If only a rough estimate is needed, a few iterations suffice.
- $x_k = \mu A^k x_0$, where $\mu$ is the normalisation constant, lies in the **Krylov subspace** $\mathcal K_{k+1}(A, x_0) = \operatorname{span}\{x_0, Ax_0, \dots, A^k x_0\}$. The power method uses only the last vector of this subspace; Krylov methods (Lanczos, Arnoldi) use all of it and converge much faster.
- For PageRank $|\lambda_2(G_\alpha)| \le \alpha$, so the error decreases at least as $\alpha^k = 0.85^k$, independently of the size of the web. The damping factor guarantees not only uniqueness, but also fast convergence.

Other eigenvalues are found with shifts and inverse iteration $x_{k+1} \propto (A - \sigma I)^{-1}x_k$, all eigenvalues with the QR algorithm (next lecture).

# Matrix decomposition: the Schur form
There is one class of matrices whose eigenvalues are easy to find: **triangular matrices**

$$
  A = \begin{pmatrix}
   \lambda_1 & * & * \\
   0 & \lambda_2 & * \\
   0 & 0 & \lambda_3 \\
  \end{pmatrix}.
$$

The eigenvalues of $A$ are $\lambda_1, \lambda_2, \lambda_3$. Why?

Because the determinant of a triangular matrix is the product of its diagonal entries:

$$
   \det(A - \lambda I) = (\lambda_1 - \lambda) (\lambda_2 - \lambda) (\lambda_3 - \lambda).
$$

- Thus computing the eigenvalues of triangular matrices is easy. Now unitary matrices come to help.
- Let $U$ be a unitary matrix, i.e. $U^* U = I$. Then

$$
   \det(A - \lambda I) = \det\big(U (U^* A U - \lambda I) U^*\big) = \det(UU^*) \det(U^* A U - \lambda I) = \det(U^* A U - \lambda I),
$$

where we have used the multiplicativity of the determinant, $\det(AB) = \det(A) \det(B)$, and $\det(UU^*) = \det I = 1$.
- It means that the matrices $U^* A U$ and $A$ have the same characteristic polynomial and the same eigenvalues.

- If we manage to make $U^* A U = T$ with $T$ **upper triangular**, we are done.
- Multiplying from the left by $U$ and from the right by $U^*$, we get the desired decomposition

$$ A = U T U^*. $$

This is the celebrated **Schur decomposition**.
- Unitary transformations do not amplify errors ($\|U\|_2 = \|U^*\|_2 = 1$), so this way of computing eigenvalues is backward stable. Whether the computed eigenvalues are accurate depends on their conditioning: see the end of the lecture.

The Schur decomposition shows why we need matrix decompositions: it represents a matrix as a product of three matrices with a convenient structure.

## Schur theorem

**Theorem (Schur).** Every matrix $A \in \mathbb C^{n\times n}$ can be written as

$$ A = U T U^*, $$

where $U$ is unitary ($U^*U = I$) and $T$ is upper triangular. The diagonal of $T$ consists of the eigenvalues of $A$.

- Unlike the eigendecomposition, the Schur decomposition exists for **every** matrix.
- The transformation $U$ is unitary, hence perfectly conditioned: the decomposition can be computed stably. This is what the QR algorithm computes.
- $T$ is similar to $A$ and triangular, so $\det(T - \lambda I) = \prod_i (t_{ii} - \lambda)$: its diagonal entries are the eigenvalues.

### Proof (induction on $n$)

1. For $n = 1$ there is nothing to prove.
2. Let $\lambda_1$ be an eigenvalue of $A$ (it exists over $\mathbb C$) and $u_1$ a unit eigenvector, $Au_1 = \lambda_1 u_1$.
3. Complete $u_1$ to a unitary matrix $U_1 = (u_1, W)$, where the columns of $W \in \mathbb C^{n\times(n-1)}$ form an orthonormal basis of the orthogonal complement of $u_1$.
4. Then

$$ U_1^* A U_1 = \begin{pmatrix} u_1^* A u_1 & u_1^* A W \\ W^* A u_1 & W^* A W \end{pmatrix} = \begin{pmatrix} \lambda_1 & * \\ 0 & A_2 \end{pmatrix}, $$

since $u_1^*Au_1 = \lambda_1$ and $W^*Au_1 = \lambda_1 W^*u_1 = 0$.

5. By induction $A_2 = U_2 T_2 U_2^*$ for $A_2 \in \mathbb C^{(n-1)\times(n-1)}$. Then $U = U_1 \begin{pmatrix} 1 & 0 \\ 0 & U_2 \end{pmatrix}$ is unitary, and $U^*AU$ is upper triangular. $\blacksquare$

**Real matrices.** If $A$ is real but has complex eigenvalues, one can stay in real arithmetic: $A = Q T Q^\top$ with orthogonal $Q$ and block upper triangular $T$ with $1\times1$ and $2\times2$ diagonal blocks (**real Schur form**).

**Note.** The proof needs an eigenvector at every step, so it is not a practical algorithm.

In [ ]:
A = rng.standard_normal((5, 5))
T, U = sla.schur(A, output='complex')
print(f'||A - U T U*|| = {np.linalg.norm(A - U @ T @ U.conj().T):.1e}')
print(f'||U* U - I||   = {np.linalg.norm(U.conj().T @ U - np.eye(5)):.1e}')
print(f'||strictly lower part of T|| = {np.linalg.norm(np.tril(T, -1)):.1e}')
print('diag(T):     ', np.sort_complex(np.diag(T)).round(4))
print('eigenvalues: ', np.sort_complex(np.linalg.eigvals(A)).round(4))

T_real, Q = sla.schur(A, output='real')
print('real Schur form:')
print(T_real.round(2))


### What the Schur form means geometrically

Write $AU = UT$ column by column. Since $T$ is upper triangular,

$$ Au_k = t_{1k}u_1 + \dots + t_{kk}u_k \in \operatorname{span}(u_1, \dots, u_k). $$

So the subspaces $V_k = \operatorname{span}(u_1,\dots,u_k)$ are **invariant**, $A V_k \subseteq V_k$, and nested: $V_1 \subset V_2 \subset \dots \subset V_n = \mathbb C^n$.

- An eigenvector spans an invariant *line*. A non-diagonalizable matrix does not have $n$ independent invariant lines, but every matrix has a nested chain of invariant subspaces of dimensions $1, 2, \dots, n$, and an orthonormal basis adapted to this chain.
- Inside $V_k$, $A$ acts as the leading $k\times k$ block of $T$: the eigenvalues $t_{11}, \dots, t_{kk}$ live in $V_k$.
- The Jordan block $\begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}$ is already in Schur form with $U = I$: $V_1$ is its only eigenline $\{(c,0)^\top\}$, and $V_2 = \mathbb C^2$.

## Normal matrices

**Theorem.** $A \in \mathbb C^{n\times n}$ is normal, $AA^* = A^*A$, if and only if

$$ A = U \Lambda U^* $$

with unitary $U$ and diagonal $\Lambda$, i.e. $A$ has an **orthonormal basis of eigenvectors**.

**Proof.**
- ($\Leftarrow$) $AA^* = U\Lambda\overline{\Lambda}U^* = U\overline{\Lambda}\Lambda U^* = A^*A$, since diagonal matrices commute.
- ($\Rightarrow$) By Schur, $A = UTU^*$. Normality of $A$ implies normality of $T$: $TT^* = T^*T$. Compare the $(1,1)$ entries of both sides for upper triangular $T$:

$$ (TT^*)_{11} = \sum_{j=1}^n |t_{1j}|^2, \qquad (T^*T)_{11} = |t_{11}|^2 \quad\Rightarrow\quad t_{12} = \dots = t_{1n} = 0. $$

The first row of $T$ is $(t_{11}, 0, \dots, 0)$; repeat the argument for the trailing $(n-1)\times(n-1)$ block, which is again normal and upper triangular. Hence $T$ is diagonal. $\blacksquare$

### Important classes of normal matrices

| matrix | definition | eigenvalues |
|---|---|---|
| Hermitian (real symmetric) | $A = A^*$ | real, since $\Lambda = \Lambda^*$ |
| skew-Hermitian | $A = -A^*$ | purely imaginary |
| unitary (orthogonal) | $A^*A = I$ | on the unit circle, $\lvert\lambda\rvert = 1$ |
| circulant | each row is a cyclic shift of the previous one | values of a polynomial at the roots of unity |

For all of them the eigenvectors can be chosen orthonormal.

**Non-normal example:** the Jordan block from the beginning of the lecture, or any triangular matrix with non-zero entries above the diagonal.

## How do we compute the Schur decomposition?

- Everything is fine, but how do we compute the Schur form?

- This will be covered in the next lecture.

## Variational principle for eigenvalues

- In many cases only the minimal or maximal eigenvalues are needed.
- For a Hermitian matrix $A$ the **Rayleigh quotient** is defined as

$$R_A(x) = \frac{(Ax, x)}{(x, x)}, \qquad x \ne 0,$$

and the maximal eigenvalue is the maximum of $R_A(x)$, the minimal eigenvalue is its minimum:

$$ \lambda_{\min} = \min_{x\ne0} R_A(x), \qquad \lambda_{\max} = \max_{x\ne0} R_A(x). $$

- Thus we can use optimisation methods to find these **extreme eigenvalues**.
- This approach will be crucial for the partial eigenvalue problem with large sparse matrices.

### Why: the Rayleigh quotient is a weighted average of the eigenvalues

Let $A = U\Lambda U^*$ and expand $x = \sum_i c_i u_i$ in the orthonormal eigenbasis. Then

$$ R_A(x) = \frac{\sum_i \lambda_i |c_i|^2}{\sum_i |c_i|^2} = \sum_i w_i\lambda_i, \qquad w_i = \frac{|c_i|^2}{\sum_j |c_j|^2} \ge 0, \quad \sum_i w_i = 1. $$

- An average lies between the smallest and the largest of the averaged numbers, and equals the largest one only if all the weight sits on it, i.e. $x$ is an eigenvector of $\lambda_{\max}$.
- Near an eigenvector, $x = u_1 + e$ with $e \perp u_1$, the weights of the other eigenvalues are $O(\|e\|_2^2)$, so $R_A(x) - \lambda_1 = O(\|e\|_2^2)$: eigenvectors are **stationary points** of $R_A$. This is why in the power method the Rayleigh quotient of a Hermitian matrix converges twice as fast as the vector, $q^{2k}$ instead of $q^k$.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 1.0]])
theta = np.linspace(0, np.pi, 400)
X = np.vstack([np.cos(theta), np.sin(theta)])     # unit vectors x(theta) as columns
R = np.einsum('ik,ij,jk->k', X, A, X)              # R_A(x) = x^T A x for each column
lam = np.linalg.eigvalsh(A)

fig, ax = plt.subplots()
ax.plot(np.degrees(theta), R, label=r'$R_A(x(\theta))$')
for l in lam:
    ax.axhline(l, color='gray', ls='--', lw=1)
ax.set_xlabel(r'angle $\theta$ of $x$, degrees')
ax.set_title(rf'$\lambda_{{\min}} = {lam[0]:.3f}$, $\lambda_{{\max}} = {lam[1]:.3f}$', fontsize=10)
ax.legend()
plt.show()


$R_A$ runs between $\lambda_{\min}$ and $\lambda_{\max}$, and it is flat at the extremes: near an eigen-direction an error in the angle changes $R_A$ only quadratically.

### Why normality matters

For a normal matrix $\|A\|_2 = \max_i|\lambda_i| = \rho(A)$ and $\|A^k\|_2 = \rho(A)^k$: the eigenvalues tell everything about the size of $A$ and its powers.

For a non-normal matrix this is false. Take

$$ A = \begin{pmatrix} 0.99 & 10 \\ 0 & 0.98 \end{pmatrix}, \qquad \rho(A) = 0.99 < 1. $$

$A^k \to 0$, but before decaying $\|A^k\|_2$ grows to about $250$ (**transient growth**): eigenvalues alone do not describe the behaviour of $x_{k+1} = Ax_k$.

In [ ]:
A = np.array([[0.99, 10.0],
              [0.0, 0.98]])
ks = np.arange(0, 1501)
norms = np.array([np.linalg.norm(np.linalg.matrix_power(A, k), 2) for k in ks])

fig, ax = plt.subplots()
ax.semilogy(ks, norms, label=r'$\|A^k\|_2$')
ax.semilogy(ks, 0.99 ** ks, 'k--', lw=1, label=r'$\rho(A)^k$')
ax.set_xlabel('$k$')
ax.legend()
plt.show()
print(f'max ||A^k|| = {norms.max():.0f} at k = {ks[norms.argmax()]}')


## Eigenvalues of non-normal matrices can be extremely sensitive

Take the $20\times20$ Jordan block $J$ with zeros on the diagonal and ones on the superdiagonal: all its eigenvalues are $0$. Change **one** entry, the bottom-left one, from $0$ to $\varepsilon$. The characteristic polynomial becomes $\pm(\lambda^{20} - \varepsilon)$, so the eigenvalues are the 20 complex roots of $\varepsilon$:

$$ |\lambda| = \varepsilon^{1/20}: \qquad \varepsilon = 10^{-10} \;\Rightarrow\; |\lambda| \approx 0.32. $$

A perturbation of size $10^{-10}$ moves the eigenvalues by $0.32$, i.e. it is amplified about $3\cdot10^{9}$ times. For a normal matrix the eigenvalues move by at most $\|\Delta A\|_2$.

**Question.** What will `np.linalg.eigvals` return for the same $J$ written in a random orthonormal basis, $QJQ^\top$?

In [ ]:
n = 20
J = np.diag(np.ones(n - 1), k=1)
J_eps = J.copy()
J_eps[-1, 0] = 1e-10
Q, _ = np.linalg.qr(rng.standard_normal((n, n)))

cases = [('J', J), ('J + eps e_n e_1^T, eps = 1e-10', J_eps), ('Q J Q^T', Q @ J @ Q.T)]
fig, ax = plt.subplots(figsize=(5, 5))
for (name, M), marker in zip(cases, ['o', 'x', '+']):
    ev = np.linalg.eigvals(M)
    ax.plot(ev.real, ev.imag, marker, ls='none', label=name)
    print(f'{name:32s} max |lambda| = {np.abs(ev).max():.2f}')
ax.set_aspect('equal')
ax.legend(fontsize=8)
plt.show()


<details><summary><b>Answer</b></summary>

Instead of 20 zeros we get a circle of radius about $0.15$. Forming $QJQ^\top$ and running the eigenvalue algorithm introduce errors of order $10^{-16}$, and they are amplified exactly like $\varepsilon$: $(10^{-16})^{1/20} \approx 0.16$. The algorithm is backward stable: it returns the exact eigenvalues of a matrix within about $10^{-16}$ of the input. It is the **problem** that is ill-conditioned. For the triangular $J$ itself LAPACK sees the zero diagonal and returns exact zeros.

The same matrix, written in another orthonormal basis, gives completely different computed eigenvalues: the computer cannot tell $J$ from its neighbours.

</details>

## Spectrum and pseudospectrum

- For linear dynamical systems given by a matrix $A$, e.g. $x_{k+1} = Ax_k$ or $\dot x = Ax$, the spectrum tells a lot about the system (stability, oscillations, ...).
- However, for **non-normal matrices** the spectrum can be unstable with respect to small perturbations, as we have just seen.
- To measure this, the notion of **pseudospectrum** has been developed.

## Pseudospectrum

We consider the union of all possible eigenvalues of all small perturbations of the matrix $A$:

$$\Lambda_{\varepsilon}(A) = \{ \lambda \in \mathbb{C}: \exists E,\ x \ne 0: (A + E) x = \lambda x, \quad \Vert E \Vert_2 \leq \varepsilon \}.$$

- An equivalent definition, convenient for computation: $\Lambda_\varepsilon(A) = \{\lambda: \sigma_{\min}(A - \lambda I) \le \varepsilon\}$, where $\sigma_{\min}$ is the smallest singular value. By the Eckart–Young theorem $\sigma_{\min}(A - \lambda I)$ is the 2-norm distance from $A - \lambda I$ to the nearest singular matrix.
- For a normal $A$, $\Lambda_\varepsilon(A)$ is exactly the union of the discs of radius $\varepsilon$ around the eigenvalues; for non-normal matrices the structure can be very different.
- More details: [Pseudospectra Gateway](http://www.cs.ox.ac.uk/pseudospectra/).

### Demo: the same eigenvalues, different pseudospectra

Both matrices have the eigenvalues $1, 2, \dots, 10$: a diagonal one (normal) and the same diagonal plus ones on the superdiagonal (non-normal). The curves are the boundaries of $\Lambda_\varepsilon$ for $\varepsilon = 0.03,\ 0.1,\ 0.3$.

In [ ]:
def sigma_min_grid(A: np.ndarray, re: np.ndarray, im: np.ndarray) -> np.ndarray:
    # smallest singular value of z I - A on the grid z = re + i im
    n = A.shape[0]
    smin = np.empty((im.size, re.size))
    for p, y in enumerate(im):
        for q, x in enumerate(re):
            smin[p, q] = np.linalg.svd((x + 1j * y) * np.eye(n) - A, compute_uv=False)[-1]
    return smin


n = 10
d = np.arange(1, n + 1, dtype=float)
cases = [('normal: diag(1, ..., 10)', np.diag(d)),
         ('non-normal: diag(1, ..., 10) + superdiagonal of ones', np.diag(d) + np.diag(np.ones(n - 1), k=1))]
re, im = np.linspace(0, 11, 221), np.linspace(-2.5, 2.5, 101)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for ax, (title, M) in zip(axes, cases):
    cs = ax.contour(re, im, np.log10(sigma_min_grid(M, re, im)), levels=np.log10([0.03, 0.1, 0.3]))
    ax.clabel(cs, fmt=lambda v: f'{10 ** v:.2g}', fontsize=8)
    ax.plot(d, np.zeros(n), 'kx')
    ax.set_aspect('equal'); ax.set_title(title, fontsize=10)
plt.show()


- Top: small discs of radius $\varepsilon$ around the eigenvalues, as for any normal matrix.
- Bottom: for the same $\varepsilon$ the regions are much larger and merge into one band: a perturbation of norm $\varepsilon$ can move the eigenvalues much further than $\varepsilon$.
- The eigenvalues are the same, the matrices behave very differently. For non-normal matrices look at the pseudospectra, not only at the spectrum.

## Summary

- Eigenvalues are the roots of $\det(A - \lambda I)$, but in practice they are never computed this way.
- Diagonalizable: $A = S\Lambda S^{-1}$; not every matrix is diagonalizable (Jordan block).
- PageRank is an eigenvector of a stochastic matrix; damping makes it unique and fast to compute.
- Gershgorin discs localise eigenvalues cheaply.
- Power method: one matrix–vector product per step, linear convergence with ratio $|\lambda_2/\lambda_1|$.
- Schur: $A = UTU^*$ exists for every matrix; $A$ is normal $\Leftrightarrow$ $T$ is diagonal $\Leftrightarrow$ $A$ has orthonormal eigenvectors.
- Variational principle: for a Hermitian $A$ the extreme eigenvalues are the extrema of the Rayleigh quotient.
- Non-normal matrices: eigenvalues can be very sensitive and do not describe the behaviour of $A^k$; pseudospectra measure this.

## Next lecture
- Review of the matrix decompositions we have considered
- A practical way to compute the QR decomposition
- An almost practical method for computing eigenvalues and eigenvectors

### Questions?